# Bronze ingestion — Swedish EV charger site selection

This notebook calls four SCB API tables and Elpriset just nu, preserves each API response as JSON in Unity Catalog **managed Delta tables**, and validates the complete requested coverage before writing. It rebuilds a fixed historical snapshot on every run; `overwrite` prevents duplicate batches. Run it as the first task of the Job.

**Scope:** 290 municipalities, SCB 2023–2025, registration months 2023-01 through 2025-12, and electricity prices in SE1–SE4 for all of 2025. SCB split the annual population series: TAB638 covers 2023–2024 and TAB5557 covers 2025, with new category codes and disclosure control. Silver must combine these two sources by municipality and year. Prices are hourly through September 30 and every 15 minutes from October 1. A full electricity refresh makes 1,460 API calls and may take several minutes. Gold must use 2025 as the decision year and 2024 as the growth baseline.


In [0]:
import json
import time
from datetime import date, datetime, timedelta, timezone
from functools import reduce
from operator import mul
from zoneinfo import ZoneInfo

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from pyspark.sql.types import (
    DateType, IntegerType, StringType, StructField, StructType, TimestampType
)

CATALOG = "db_labb2"
YEARS = (2023, 2024, 2025)
PRICE_START = date(2025, 1, 1)
PRICE_END = date(2025, 12, 31)
AREAS = ("SE1", "SE2", "SE3", "SE4")
SCB_BASE = "https://statistikdatabasen.scb.se/api/v2/tables"
PRICE_BASE = "https://www.elprisetjustnu.se/api/v1/prices"

def http_session():
    """Create an HTTP session that retries transient API failures."""
    session = requests.Session()
    retry = Retry(
        total=5, connect=5, read=5,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    session.mount("https://", HTTPAdapter(max_retries=retry))
    return session

def response_json(session, url, params=None):
    """Request a JSON document and raise an error for unsuccessful responses."""
    response = session.get(url, params=params, timeout=(10, 90))
    response.raise_for_status()
    return response.json()

print("Target catalog:", CATALOG)


## SCB: fetch and validate raw JSON-stat responses

Each row in a Bronze table is one original API batch. `payload_json` contains the entire response, including labels used later in Gold. No JSON-stat values are cleaned or expanded here.


In [0]:
def dimension_codes(payload, dimension_id):
    """Return JSON-stat category codes in their declared value-array order."""
    index = payload["dimension"][dimension_id]["category"]["index"]
    return [code for code, _ in sorted(index.items(), key=lambda item: item[1])] if isinstance(index, dict) else index

def validate_jsonstat(payload, expected_regions, expected_times, expected_codes):
    """Check requested dimensions and value count; return the number of values."""
    ids = payload["id"]
    assert set(expected_codes).issubset(ids) and "Region" in ids and "Tid" in ids
    assert set(dimension_codes(payload, "Region")) == set(expected_regions)
    assert set(dimension_codes(payload, "Tid")) == set(expected_times)
    for dimension_id, codes in expected_codes.items():
        assert set(dimension_codes(payload, dimension_id)) == set(codes), dimension_id
    expected_count = reduce(mul, payload["size"], 1)
    assert len(payload["value"]) == expected_count
    assert all(value is not None for value in payload["value"]), "SCB returned missing values"
    return expected_count

SCB_TABLES = (
    ("TAB638", "BE0101N1", {}, ["2023", "2024"], 290 * 2),
    ("TAB5557", "000007ME", {"Civilstand": ["SC"], "Alder": ["TotSA"], "Kon": ["TotSa"]},
     ["2025"], 290),
    ("TAB3277", "TK1001AA", {"Drivmedel": ["120", "140"]},
     [f"{y}M{m:02d}" for y in YEARS for m in range(1, 13)], 290 * 2 * 12 * len(YEARS)),
    ("TAB1278", "TK1001AC", {"Fordonsslag": ["10"]},
     [str(y) for y in YEARS], 290 * len(YEARS)),
)

SCB_SCHEMA = StructType([
    StructField("source", StringType(), False),
    StructField("table_id", StringType(), False),
    StructField("batch_no", IntegerType(), False),
    StructField("fetched_at_utc", TimestampType(), False),
    StructField("payload_json", StringType(), False),
])

with http_session() as session:
    metadata = response_json(session, f"{SCB_BASE}/TAB638/metadata", {"lang": "sv"})
    kommun_codes = sorted(code for code in metadata["dimension"]["Region"]["category"]["index"]
                          if len(code) == 4 and code.isdigit())
    assert len(kommun_codes) == 290 and len(set(kommun_codes)) == 290

    for table_id, content_code, filters, times, expected_rows in SCB_TABLES:
        table_metadata = response_json(session, f"{SCB_BASE}/{table_id}/metadata", {"lang": "sv"})
        for dimension_id, requested in {"ContentsCode": [content_code], "Tid": times, **filters}.items():
            available = set(dimension_codes(table_metadata, dimension_id))
            assert set(requested) <= available, (table_id, dimension_id, sorted(set(requested) - available))
        regions_available = set(dimension_codes(table_metadata, "Region"))
        assert set(kommun_codes) <= regions_available, (table_id, "missing municipalities")
        records = []
        observed_rows = 0
        for offset in range(0, len(kommun_codes), 25):
            batch = kommun_codes[offset:offset + 25]
            params = {
                "lang": "sv",
                "valueCodes[ContentsCode]": content_code,
                "valueCodes[Region]": ",".join(batch),
                "valueCodes[Tid]": ",".join(times),
                **{f"valueCodes[{dimension}]": ",".join(codes)
                   for dimension, codes in filters.items()},
            }
            payload = response_json(session, f"{SCB_BASE}/{table_id}/data", params)
            observed_rows += validate_jsonstat(
                payload, batch, times, {"ContentsCode": [content_code], **filters}
            )
            records.append((
                "SCB", table_id, offset // 25 + 1, datetime.now(timezone.utc),
                json.dumps(payload, ensure_ascii=False, separators=(",", ":")),
            ))

        assert len(records) == 12 and observed_rows == expected_rows, (table_id, observed_rows)
        target = f"{CATALOG}.bronze.scb_{table_id.lower()}"
        df = spark.createDataFrame(records, SCB_SCHEMA)
        df.write.format("delta").mode("overwrite").saveAsTable(target)
        assert spark.table(target).count() == 12
        print(f"{target}: 12 raw responses, {observed_rows} JSON-stat values")


## Electricity: fetch and validate each date and area

The source returns local time offsets. Bronze keeps those strings unchanged. Silver converts interval timestamps to UTC and computes duration.


In [0]:
PRICE_SCHEMA = StructType([
    StructField("source", StringType(), False),
    StructField("price_date", DateType(), False),
    StructField("price_area", StringType(), False),
    StructField("source_url", StringType(), False),
    StructField("fetched_at_utc", TimestampType(), False),
    StructField("payload_json", StringType(), False),
])

def days_inclusive(start, end):
    """Yield every calendar date from start through end, inclusive."""
    current = start
    while current <= end:
        yield current
        current += timedelta(days=1)

def validate_price_payload(payload, day, area):
    """Validate interval lengths and continuous coverage of a Swedish local day.

    The API has one incorrect time_end at the autumn clock change on
    2025-10-26. Keep the original JSON, but validate that interval using
    its start time plus 15 minutes.
    """
    assert isinstance(payload, list) and payload, (day, area)
    expected_minutes = 60 if day < date(2025, 10, 1) else 15
    intervals = []

    for item in payload:
        assert {"time_start", "time_end", "SEK_per_kWh"}.issubset(item)
        start = datetime.fromisoformat(item["time_start"])
        end = datetime.fromisoformat(item["time_end"])
        assert start.tzinfo is not None and end.tzinfo is not None

        start_utc = start.astimezone(timezone.utc)
        end_utc = end.astimezone(timezone.utc)
        expected_end_utc = start_utc + timedelta(minutes=expected_minutes)

        if end_utc != expected_end_utc:
            assert (
                day == date(2025, 10, 26)
                and item["time_start"] == "2025-10-26T02:45:00+02:00"
                and item["time_end"] == "2025-10-26T03:00:00+01:00"
                and expected_end_utc
                    == datetime(2025, 10, 26, 1, tzinfo=timezone.utc)
            ), (day, area, item)

        intervals.append((start_utc, expected_end_utc))
        assert item["SEK_per_kWh"] is not None

    intervals.sort()
    sweden = ZoneInfo("Europe/Stockholm")
    first = datetime.combine(
        day, datetime.min.time(), tzinfo=sweden
    ).astimezone(timezone.utc)
    last = datetime.combine(
        day + timedelta(days=1), datetime.min.time(), tzinfo=sweden
    ).astimezone(timezone.utc)

    assert intervals[0][0] == first and intervals[-1][1] == last, (day, area)
    assert all(
        left[1] == right[0]
        for left, right in zip(intervals, intervals[1:])
    ), (day, area)
    assert len(intervals) == (last - first).total_seconds() / (
        expected_minutes * 60
    )

price_records = []
with http_session() as session:
    for day in days_inclusive(PRICE_START, PRICE_END):
        for area in AREAS:
            url = f"{PRICE_BASE}/{day:%Y}/{day:%m-%d}_{area}.json"
            payload = response_json(session, url)
            validate_price_payload(payload, day, area)
            price_records.append((
                "Elpriset just nu",
                day,
                area,
                url,
                datetime.now(timezone.utc),
                json.dumps(payload, ensure_ascii=False, separators=(",", ":")),
            ))
            time.sleep(0.2)

expected_days = (PRICE_END - PRICE_START).days + 1
assert len(price_records) == expected_days * len(AREAS) == 1460
assert len({(record[1], record[2]) for record in price_records}) == 1460

target = f"{CATALOG}.bronze.electricity_price_raw"
(
    spark.createDataFrame(price_records, PRICE_SCHEMA)
    .write.format("delta")
    .mode("overwrite")
    .saveAsTable(target)
)
assert spark.table(target).count() == 1460
print(f"{target}: 1460 raw API responses")


## Verify managed tables and coverage


In [0]:
for table_id, *_ in SCB_TABLES:
    table = f"{CATALOG}.bronze.scb_{table_id.lower()}"
    print(table, spark.table(table).count())
print(f"{CATALOG}.bronze.electricity_price_raw",
      spark.table(f"{CATALOG}.bronze.electricity_price_raw").count())

# The table detail should show type MANAGED and provider delta.
display(spark.sql(f"DESCRIBE DETAIL {CATALOG}.bronze.scb_tab638")
    .select("name", "format", "location"))
